# Week 8 Exercise — Emmanuel

In [ ]:
from datasets import load_dataset, Dataset
import tiktoken
import os
import uuid
from dotenv import load_dotenv

load_dotenv(override=True)
import pandas as pd
from openai import OpenAI
from litellm import completion
from pydantic import BaseModel, Field
import gradio as gr
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [ ]:
MODEL = "gpt-4.1-nano"
db_name = "customer_suppport"
openai = OpenAI()

# Default system prompt — NO RAG. Customer support only; decline out-of-scope.
DEFAULT_SYSTEM_PROMPT = """
You are a customer support assistant for an e-commerce platform. You ONLY address customer support conversations related to:
- Orders (placement, confirmation, status, payment, invoice)
- Shipping and delivery
- Returns, refunds, cancellations, exchanges
- Login and account issues
- Product information, availability, pricing
- Warranty and product registration

If the user's message is OUT OF SCOPE (e.g., general chat, trivia, unrelated topics, jokes, or anything not related to e-commerce customer support), you MUST politely decline. Say something like: "I'm sorry, I can only help with customer support questions about orders, shipping, returns, accounts, and similar topics. I'm not able to assist with that. Is there a customer support issue I can help you with?"

For in-scope greetings or simple questions, respond briefly and helpfully. For detailed support issues that need resolution, you must use the forward_to_triage tool to look up solutions in the knowledge base.
"""
SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT

In [ ]:
# List all columns

dataset = load_dataset("NebulaByte/E-Commerce_Customer_Support_Conversations")

split = list(dataset.keys())[0]
columns = dataset[split].column_names
print("Columns:", columns)

In [ ]:
# Count unique issue categories and issue areas
split = list(dataset.keys())[0]
data = dataset[split]

issue_areas = set(data["issue_area"])
issue_categories = set(data["issue_category"])

print(f"Issue areas: {len(issue_areas)}")
print(f"Issue categories: {len(issue_categories)}")
print(f"\nIssue areas: {sorted(issue_areas)}")
print(f"\nIssue categories: {sorted(issue_categories)}")

In [ ]:
# Tokenize with tiktoken (gpt-4o-mini encoding)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
split = list(dataset.keys())[0]
data = dataset[split]

total_tokens = 0
for row in data:
    text = str(row.get("conversation", ""))
    total_tokens += len(encoding.encode(text))

print(f"Number of tokens: {total_tokens:,}")

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

split = list(dataset.keys())[0]
data = dataset[split]
columns = data.column_names


documents = []


for idx, row in enumerate(data):
    # Create rich text content (what gets embedded)
    # Focus on the most important fields for semantic search
    conversation_text = row.get('conversation', '')
    
    # Create a comprehensive searchable text
    page_content = f"""
    Issue Area: {row.get('issue_area', 'N/A')}
    Issue Category: {row.get('issue_category', 'N/A')} - {row.get('issue_sub_category', 'N/A')}
    Product: {row.get('product_category', 'N/A')} - {row.get('product_sub_category', 'N/A')}
    Customer Sentiment: {row.get('customer_sentiment', 'N/A')}
    Issue Complexity: {row.get('issue_complexity', 'N/A')}
    Agent Experience: {row.get('agent_experience_level', 'N/A')}
    
    Conversation:
    {conversation_text}
    """
    
    # Create metadata (for filtering)
    metadata = {
        "issue_area": row.get('issue_area', ''),
        "issue_category": row.get('issue_category', ''),
        "issue_sub_category": row.get('issue_sub_category', ''),
        "customer_sentiment": row.get('customer_sentiment', ''),
        "product_category": row.get('product_category', ''),
        "product_sub_category": row.get('product_sub_category', ''),
        "issue_complexity": row.get('issue_complexity', ''),
        "agent_experience_level": row.get('agent_experience_level', ''),
        "doc_id": str(uuid.uuid4()),  # Unique ID for each document
        "conversation_length": len(conversation_text.split()) if conversation_text else 0
    }
    
    # Create Document object
    doc = Document(
        page_content=page_content,
        metadata=metadata
    )
    documents.append(doc)


print(f"Created {len(documents)} documents")


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("First chunk (preview):\n", chunks[0].page_content[:400], "...")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
retriever = vectorstore.as_retriever()
collection = vectorstore._collection
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
# DEFAULT_SYSTEM_PROMPT and SYSTEM_PROMPT are defined in the config cell (cell 2)

In [ ]:
# Triage Agent — Classifies incoming tickets (Week 8 Day 5 style)
import logging

TICKET_CATEGORIES = [
    "Cancellations and returns",
    "Login and Account",
    "Order",
    "Shipping",
    "Shopping",
    "Warranty",
]


class TriageResult(BaseModel):
    """Classification result from the Triage Agent."""
    category: str = Field(
        description="The issue area: Cancellations and returns, Login and Account, Order, Shipping, Shopping, or Warranty"
    )
    confidence: str = Field(
        description="How confident the classification is: high, medium, or low"
    )


class TriageAgent:
    """Agent that classifies incoming support tickets into issue areas."""
    name = "Triage Agent"
    SYSTEM_PROMPT = """You are a customer support triage specialist. Classify each incoming ticket into exactly one of these issue areas:
- Cancellations and returns: refunds, returns, exchanges, order cancellations
- Login and Account: login issues, account verification, password reset, account settings
- Order: order placement, confirmation, status, payment, invoice
- Shipping: delivery, tracking, shipping options, delivery speed, pickup
- Shopping: product info, availability, pricing, discounts, product search
- Warranty: warranty claims, warranty terms, product registration, extended warranty

Respond with the category and your confidence (high/medium/low)."""

    def __init__(self):
        self.log("Triage Agent is initializing")
        self.log("Triage Agent is ready")

    def log(self, message: str):
        logging.info(f"[{self.name}] {message}")

    def classify(self, ticket_text: str, history: list[dict] = None) -> TriageResult:
        """Classify an incoming ticket into one of the 6 issue areas."""
        self.log("Classifying incoming ticket")
        history = history or []
        context = ""
        if history:
            recent = history[-5:]  
            context = "Conversation so far:\n" + "\n".join(
                f"{'User' if m.get('role') == 'user' else 'Assistant'}: {m.get('content', '')[:200]}"
                for m in recent
            ) + "\n\n"
        user_prompt = f"{context}Current ticket/message:\n{ticket_text}"
        response = completion(
            model=MODEL,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            response_format=TriageResult,
        )
        reply = response.choices[0].message.content
        result = TriageResult.model_validate_json(reply)
        if result.category not in TICKET_CATEGORIES:
            result.category = "Shopping"  # fallback
        self.log(f"Classified as: {result.category} (confidence: {result.confidence})")
        return result


triage_agent = TriageAgent()

In [ ]:
# Resolution Agent — Solves issues using knowledge base, fine-tuned by LLM
class ResolutionAgent:
    """
    Attempts to solve common issues using the knowledge base.
    Context comes from the vector store; the final result is fine-tuned by the LLM.
    """
    name = "Resolution Agent"

    DRAFT_SYSTEM = """You are a customer support resolution specialist. Your job is to draft a resolution based ONLY on the provided knowledge base context.

Given the customer's issue and relevant past support conversations, extract and synthesize a draft resolution.
- Base your draft strictly on what the context says; do not invent information
- Include specific steps, policies, or solutions mentioned in the context
- If the context does not contain enough information, note what is missing
- Be factual and concise; we will polish the tone in a later step"""

    REFINE_SYSTEM = """You are a customer support expert polishing a draft resolution into a final response.

You will receive:
1. The customer's original message
2. The triage category
3. A draft resolution based on the knowledge base

Your job: Refine the draft into a warm, clear, professional response for the customer.
- Use a friendly, empathetic tone
- Ensure the response flows naturally and is easy to follow
- Add appropriate greetings/closings if helpful
- Do NOT add information not in the draft; only polish and improve clarity
- If the draft says information is missing, politely offer to escalate or get more details"""

    def __init__(self):
        self.log("Resolution Agent is initializing")
        self.log("Resolution Agent is ready")

    def log(self, message: str):
        logging.info(f"[{self.name}] {message}")

    def _format_context_for_draft(self, chunks) -> str:
        """Format retrieved chunks for the draft step."""
        def _source(chunk):
            m = chunk.metadata
            parts = [m.get("issue_sub_category", ""), m.get("product_category", "")]
            return " | ".join(p for p in parts if p) or "Support conversation"
        return "\n\n---\n\n".join(
            f"[Source: {_source(c)}]\n{c.page_content}" for c in chunks
        )

    def draft_resolution(self, question: str, chunks: list, triage_category: str) -> str:
        """Draft a resolution from the knowledge base context."""
        self.log("Drafting resolution from knowledge base")
        if not chunks:
            return "No relevant context found in the knowledge base. Unable to draft a resolution."
        context = self._format_context_for_draft(chunks)
        user_msg = f"""Triage category: {triage_category}

Customer's issue:
{question}

Knowledge base context (past support conversations):
{context}

Provide a draft resolution based on the above context. Be specific and cite what the context says."""

        response = completion(
            model=MODEL,
            messages=[
                {"role": "system", "content": self.DRAFT_SYSTEM},
                {"role": "user", "content": user_msg},
            ],
        )
        draft = response.choices[0].message.content
        self.log("Draft resolution complete")
        return draft

    def refine_response(self, question: str, triage_category: str, draft: str, history: list[dict] = None) -> str:
        """Fine-tune the draft into a polished customer-facing response."""
        self.log("Refining response with LLM")
        history = history or []
        conv_context = ""
        if history:
            recent = history[-4:]
            conv_context = "\nConversation so far:\n" + "\n".join(
                f"{'Customer' if m.get('role') == 'user' else 'Agent'}: {m.get('content', '')[:150]}"
                for m in recent
            ) + "\n\n"
        user_msg = f"""Triage category: {triage_category}
{conv_context}Current customer message: {question}

Draft resolution:
{draft}

Produce the final polished response for the customer."""

        response = completion(
            model=MODEL,
            messages=[
                {"role": "system", "content": self.REFINE_SYSTEM},
                {"role": "user", "content": user_msg},
            ],
        )
        final = response.choices[0].message.content
        self.log("Resolution refined and ready")
        return final

    def resolve(self, question: str, chunks: list, triage_category: str, history: list[dict] = None) -> str:
        """
        Attempt to resolve the issue: draft from knowledge base, then refine with LLM.
        """
        draft = self.draft_resolution(question, chunks, triage_category)
        return self.refine_response(question, triage_category, draft, history or [])


resolution_agent = ResolutionAgent()

In [ ]:
print(triage_agent.classify("Hi Tom, I placed an order for an Electric Kettle from your website, but I noticed that the price for the same product is different on your website now. Can you explain why?"))

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )


class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# format_context and chat are defined in the Gradio cell below


def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant support conversations."""
    message = f"""
You are a customer support agent helping users with their issues.
You are about to look up similar past conversations in a Knowledge Base to help answer the user's current question.

This is the conversation history so far:
{history}

And this is the user's current question/issue:
{question}

Respond only with a single, refined search query that will find the most relevant past support conversations.
Focus on:
- The core issue type (verification, login, product issue, etc.)
- The specific product or feature mentioned (OTG, mobile number, email, etc.)
- Key technical terms from the user's description
- Keep it VERY short and specific (3-8 words ideally)

Examples:
Original: "I'm trying to log in but it keeps asking me to verify my number and I don't have access to that phone anymore"
Refined: "mobile number verification without access"

Original: "My OTG is not heating properly, what should I do?"
Refined: "OTG heating problems"

Original: "I forgot which email I used to register, can you help me find it?"
Refined: "find registered email address"

IMPORTANT: Respond ONLY with the refined search query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]


# RAG retrieval (defined here so fetch_context is available before run_triage_and_resolution)
RETRIEVAL_K = 10
EMBEDDING_MODEL = "text-embedding-3-large"


def fetch_context_unranked(question):
    query = openai.embeddings.create(model=EMBEDDING_MODEL, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
# RAG + routing section (Resolution Agent handles RAG; make_rag_messages no longer used)

In [ ]:
# Tool definition for routing: forward_to_triage triggers RAG + Triage + Resolution
FORWARD_TO_TRIAGE_TOOL = {
    "type": "function",
    "function": {
        "name": "forward_to_triage",
        "description": "Forward the customer's support issue to the triage agent. Use this when the user has a customer support question that requires looking up solutions in the knowledge base (orders, shipping, returns, login, warranty, product issues, etc.). Do NOT use for out-of-scope messages.",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {
                    "type": "string",
                    "description": "Brief reason why this needs triage (e.g., 'order status question', 'return policy inquiry')"
                }
            },
            "required": ["reason"]
        }
    }
}


def run_triage_and_resolution(question: str, history: list[dict]) -> tuple[str, list, TriageResult]:
    """
    Run Triage → RAG → Resolution. Called when forward_to_triage tool is invoked.
    Triage classifies first, then we fetch RAG. Triage forwards to Resolution with RAG context.
    """
    # 1. Triage classifies (no RAG yet)
    triage_result = triage_agent.classify(question, history)
    # 2. Fetch RAG with triage category for better retrieval
    query = rewrite_query(question, history)
    query_with_triage = f"{triage_result.category} {query}".strip()
    print(f"[RAG] Fetching context for: {query_with_triage}")
    chunks = fetch_context(query_with_triage)
    print(f"[Triage] Forwarding to Resolution with {len(chunks)} chunks: {triage_result.category}")
    # 3. Resolution uses same RAG to find solution
    answer = resolution_agent.resolve(question, chunks, triage_result.category, history)
    return answer, chunks, triage_result

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list, TriageResult | None]:
    """
    Tool-based routing: NO RAG by default.
    - Uses tools to decide if message needs forward_to_triage
    - If tool called: RAG → Triage → Resolution
    - If no tool: default response (in-scope or out-of-scope decline)
    """
    # Build messages for router
    messages = [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}]
    for m in history:
        messages.append({"role": m["role"], "content": m["content"]})
    messages.append({"role": "user", "content": question})

    # Call OpenAI with tools
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[FORWARD_TO_TRIAGE_TOOL],
        tool_choice="auto",
    )
    choice = response.choices[0]
    message = choice.message

    # Check for tool call
    tool_calls = getattr(message, "tool_calls", None) or []
    if choice.finish_reason == "tool_calls" or tool_calls:
        for tc in tool_calls:
            if getattr(tc.function, "name", None) == "forward_to_triage":
                return run_triage_and_resolution(question, history)

    # No tool call: use LLM response directly (in-scope or decline)
    answer = getattr(message, "content", None) or ""
    if not answer:
        answer = "I'm not sure how to respond. Could you rephrase your question?"
    return answer, [], None

In [ ]:
def format_context(docs, triage_result=None):
    """Format retrieved chunks as HTML for the context panel. Shows triage category if provided."""
    lines = []
    if not docs and not triage_result:
        return '<div style="background: #f8fafc; padding: 10px; border-radius: 4px;"><em>Direct response — no knowledge base lookup (out-of-scope or simple query)</em></div>'
    if triage_result:
        badge_color = {"high": "#22c55e", "medium": "#eab308", "low": "#f97316"}.get(triage_result.confidence, "#6b7280")
        lines.append(
            f'<div style="background: #f0f9ff; border-left: 4px solid {badge_color}; padding: 10px; margin-bottom: 12px; border-radius: 4px;">'
            f'<strong>🎫 Triage:</strong> <span style="font-weight: 600;">{triage_result.category}</span> '
            f'(confidence: {triage_result.confidence})</div>'
        )
    if not docs:
        return ("\n".join(lines) + "<p><em>No context retrieved.</em></p>") if lines else "<p><em>No context retrieved.</em></p>"
    lines.append("<h3>Relevant Support Conversations</h3>")
    for i, d in enumerate(docs, 1):
        issue = d.metadata.get("issue_sub_category", "")
        product = d.metadata.get("product_sub_category", "")
        sentiment = d.metadata.get("customer_sentiment", "")
        agent_level = d.metadata.get("agent_experience_level", "")
        src_parts = [f"Issue: {issue}"] if issue else []
        if product:
            src_parts.append(f"Product: {product}")
        if sentiment:
            src_parts.append(f"Sentiment: {sentiment}")
        if agent_level:
            src_parts.append(f"Agent: {agent_level}")
        src = " | ".join(src_parts) if src_parts else f"Conversation {i}"
        lines.append(f"<p><strong>Source {i}:</strong> {src}</p>")
        lines.append(f"<pre style='white-space: pre-wrap; background-color: #f5f5f5; padding: 10px; border-radius: 5px;'>{d.page_content[:2000]}</pre>")
    return "\n".join(lines)


def chat(history):
    """Append assistant reply and return updated history and context HTML."""
    if not history:
        return history, "*No message.*"
    last = history[-1]
    if last.get("role") != "user":
        return history, "*Send a message first.*"
    user_msg = last["content"]
    prior = history[:-1]
    answer, context_docs, triage_result = answer_question(user_msg, prior)
    history = history + [{"role": "assistant", "content": answer}]
    return history, format_context(context_docs, triage_result)


def main():
    def submit(message, history):
        history = history or []
        if not message or not message.strip():
            return "", history
        return "", history + [{"role": "user", "content": message.strip()}]

    with gr.Blocks(title="Customer Support Assistant", theme=gr.themes.Soft()) as ui:
        gr.Markdown("# Customer Support Assistant\nAsk questions about e-commerce customer support from the dataset.")
        with gr.Row():
            with gr.Column(scale=1):
                chatbot = gr.Chatbot(label="Conversation", height=500, type="messages", show_copy_button=True)
                msg = gr.Textbox(placeholder="Ask anything about customer support...", show_label=False)
            with gr.Column(scale=1):
                context_md = gr.Markdown(value="*Retrieved context will appear here.*", elem_id="context", height=500)
        msg.submit(submit, inputs=[msg, chatbot], outputs=[msg, chatbot]).then(
            chat, inputs=chatbot, outputs=[chatbot, context_md]
        )
    ui.launch(inbrowser=True)


main()

In [ ]:



class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]